In [0]:
%sql

CREATE VIEW IF NOT EXISTS dev.gold.countries_stats_vw AS(
  SELECT 
    country,
    date_trunc("DD", order_timestamp) order_date,
    count( order_id ) order_count,
    sum( quantity ) books_count
  FROM dev.silver.customers_orders
  GROUP BY country, date_trunc("DD", order_timestamp)
)

In [0]:
%sql
SELECT 
  *
FROM dev.gold.countries_stats_vw
where country = 'France'

In [0]:
from pyspark.sql import functions as F
query = (
    spark.readStream.table('dev.silver.books_sales')
            .withWatermark("order_timestamp", "10 minutes")
            .groupBy(
                F.window("order_timestamp", "5 minutes").alias("time"),
                "author"
            ).agg(
                F.count("order_id").alias("orders_count"),
                F.avg("quantity").alias("avg_quantity")
            )
        .writeStream
            .option("checkpointLocation", f"dbfs:/Volumes/dev/pro_landing_zone/checkpoints/authors_stats")
            .trigger(availableNow=True)
            .toTable("dev.gold.authors_stats")
).awaitTermination()

In [0]:
%sql
select 
  *
from dev.gold.authors_stats